# Supervised Machine Learning 2

In [1]:
# Step 1: Import Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Step 2: Load Dataset
insurance_data = pd.read_csv("insurance.csv")

# Step 3: Explore the Data (optional but recommended)
print(insurance_data.head())
print(insurance_data.shape)
print(insurance_data.dtypes)
print(insurance_data.isnull().sum())

   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520
(1338, 7)
age           int64
sex             str
bmi         float64
children      int64
smoker          str
region          str
charges     float64
dtype: object
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64


In [2]:
# One Hot Encoding

X = insurance_data.drop(columns=["charges"])
y = insurance_data["charges"]

X = pd.get_dummies(X, columns=["region"], drop_first=True)# to aviod dummy variable trap

X["sex"] = X["sex"].map({"female": 1, "male": 0})
X["smoker"] = X["smoker"].map({"yes": 1, "no": 0})

X.head()

,age,sex,bmi,children,smoker,region_northwest,region_southeast,region_southwest
0,19,1,27.900,0,1,False,False,True
1,18,0,33.770,1,0,False,True,False
2,28,0,33.000,3,0,False,True,False
3,33,0,22.705,0,0,True,False,False
4,32,0,28.880,0,0,True,False,False


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
print("r2 score: ", r2)


r2 score:  0.7835929767120724


## The Core Insight

| Encoding | Columns Used | Trap? |
|---|---|---|
| Label Encoding (0/1) | 1 column | ❌ No trap |
| Dummy (`drop_first=False`) | N columns | ✅ Trap present |
| Dummy (`drop_first=True`) | N-1 columns | ❌ No trap |

In [4]:
# derived features & interaction features
X["age_smoker"] = X["age"] * X["smoker"]
X["bmi_smoker"] = X["bmi"] * X["smoker"]
X.head()


,age,sex,bmi,children,smoker,region_northwest,region_southeast,region_southwest,age_smoker,bmi_smoker
0,19,1,27.900,0,1,False,False,True,19,27.9
1,18,0,33.770,1,0,False,True,False,0,0.0
2,28,0,33.000,3,0,False,True,False,0,0.0
3,33,0,22.705,0,0,True,False,False,0,0.0
4,32,0,28.880,0,0,True,False,False,0,0.0


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
print("r2 score: ", r2)


r2 score:  0.865231697953168


## Derived Features

**Derived features** are new columns created by combining 2 or more 
existing columns to capture hidden relationships that affect the target.

---

## The Core Rule

> Every column in the combination must **individually affect the target**
> AND their combination must make **logical real-world sense**

---

## Examples (Insurance Dataset)

| Derived Feature | Combination | Logic |
|---|---|---|
| `age_smoker` | `age × smoker` | Older smokers = much higher risk |
| `bmi_smoker` | `bmi × smoker` | Obese smokers = much higher risk |

---

## What Makes a Valid Derived Feature?

✅ Both columns individually affect the target  
✅ Combination makes real-world sense  
❌ `sex × children` → no logical relation to charges  
❌ Random combinations just add noise  

---

## Drop or Keep Original Columns?

| Case | Example | Drop Original? |
|---|---|---|
| **Relationship** (new info) | `bmi × smoker` → `bmi_smoker` | ❌ Keep all 3 |
| **Replacement** (same info, new format) | `sex` → mapped to 0/1 | ✅ Drop original |

> **New info = Keep old, Same info new format = Drop old**

---

## Key Insight

> Feature Engineering is **not complex mathematics** —  
> it is **human understanding + common sense** about the data.  
> A domain expert knows which combinations make real-world sense.

In [6]:
#scaling numeric features
# to avoid one of the feature dominating the model due to its scale
# salary=1000000, age=30, the model will give more importance to salary than age because of its scale


In [7]:
# underfit & overfit
# R2 trainign score is low ,R2 testing score is low => underfitting
# R2 training score is high, R2 testing score is low => overfitting
y_train_pred = model.predict(X_train)
r2_train = r2_score(y_train, y_train_pred)
print("R2 training score: ", r2_train)
print("R2 testing score: ", r2)



R2 training score:  0.8340713711218875
R2 testing score:  0.865231697953168
